# 3D luminosity upper limit — GW follow-up

Two-part pipeline for one GW source:

1. **Part 1 — Data preparation**: read the GW skymap and the real DL3 data, build the WCS
   geometry, the 95% GW mask, the per-bin distance CDF table, the estimators, and the
   real-data $\Lambda$ — then save and *validate* everything the simulations need into one
   `.pkl`. Nothing here is Monte Carlo.
2. **Part 2 — Simulations**: background-only realisations (for the observation's
   significance), then the two iterative (bisection) upper limits — 2D flux and 3D
   luminosity — which share `gwuls.simulate._bisect_ul`. Each bisection step's `n_sim`
   realisations can run in parallel across CPU cores, and `n_sim` itself can ramp up only
   once the bracket has narrowed — see `N_JOBS` / `USE_DYNAMIC_N_SIM` in the parameter cell.

See the module docstring in [`gwuls/simulate.py`](../gwuls/simulate.py) for the statistical
method (the shared test statistic `Lambda = max over the GW region of TS + 2 ln p_GW`, and the
flux ↔ luminosity conversions), and [`README.md`](../README.md) for the cluster/local workflow.

## Part 1 — Data preparation

### Import packages

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

import os, json, pickle, logging, copy
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import healpy as hp
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
from scipy.stats import norm
from scipy.optimize import curve_fit, root_scalar
from scipy.ndimage import gaussian_filter
from gammapy import __version__ as gammapy_version, __path__ as gammapy_path
from gammapy.irf import load_irf_dict_from_file
from gammapy.maps import MapAxis, WcsGeom, Map
from gammapy.modeling.models import (
    PointSpatialModel, PowerLawSpectralModel,
    ConstantTemporalModel, SkyModel, Models
)
from gammapy.data import DataStore, Observation, FixedPointingInfo, PointingMode
from gammapy.makers import MapDatasetMaker, SafeMaskMaker, RingBackgroundMaker
from gammapy.datasets import Dataset, Datasets, MapDataset, MapDatasetOnOff
from gammapy.estimators import ExcessMapEstimator, TSMapEstimator
from gammapy.estimators.map.excess import convolved_map_dataset_counts_statistics, _get_convolved_maps
from gammapy.stats.fit_statistics import cash

import sys
sys.path.insert(0, "..")
from gwuls import utils, simulate, plotting, slurm
from gwuls import paths as gwpaths
from importlib import reload
reload(utils); reload(simulate); reload(plotting); reload(slurm)

# --- Config ---
warnings.filterwarnings("ignore", category=RuntimeWarning, append=True)
warnings.filterwarnings("ignore", message=".*outside valid IRF map range.*")
import lal
logging.getLogger("gammapy").setLevel(logging.ERROR)
import ligo.skymap.plot
from ligo.skymap.io.fits import read_sky_map

# --- Setup ---
observing_location = EarthLocation.of_site("Roque de los Muchachos")
gwpaths.ensure_dirs()

print(f"Using Gammapy version: {gammapy_version}\nIn path: {gammapy_path[0]}")

##### <span style="color:blue">Main parameters</span>
* `source_name` Source name
* `dir_dl3` DL3 directory
* `e_min`, `e_max` and `e_bins`: Maximum, minimum energies and bins per decade
* `obs_id`: The run numbers to be used
* `bkg_type` If you want to use `baccmod` or `pybkgmodel`
* `e_min`, `e_max` and `e_bins`: Maximum, minimum offset, and total bins

In [ ]:
source_name = "S240615dg"
source_coord = SkyCoord(ra=7.53, dec=45.81, unit="deg")

dir_dl3 = os.path.join(
    "/fefs/aswg/workspace/juan.jimenez/data/real", "stereo", source_name, "v0.6.1",
    "GammaDiffuse", "prod_standard", "DL3", "gh_dyn90"
)

obs_ids = [17821, 17822, 17823, 17824, 17825]

# Tool used for background computation
bkg_type = "pybkgmodel" # baccmod or pybkgmodel

# Binning: energy and offset
e_min, e_max, e_bins = 0.6 * u.TeV, 20 * u.TeV, 4.5
o_min, o_max, o_bins = 0 * u.deg, 2.5 * u.deg, 8 # Offset binning
correlation_radius = 0.1 * u.deg

Other secundary parameters

In [ ]:
# --- Flags ---
USE_DIRAC_DELTA   = False
USE_ITERATIVE_ULS = True
OVERWRITE_RESULTS = True
RUN_CROSSCHECKS   = True          # spectral + sampler validation cells

# --- Grid & Energy Geometry ---
binsz                 = 0.08
resolution_hp_to_grid = int(2048 / 1)         # HEALPix -> grid resolution
size_fov              = 2.5  * u.deg
geom_width            = (size_fov.value * 1.43,) * 2
e_t_min, e_t_max, e_t_bins = 0.05 * u.TeV, 100 * u.TeV, 6
ring_r_in, ring_width = 0.3  * u.deg, 0.2  * u.deg
exclusion_mask = None

# --- Source & Skymap ---
delta_coord = SkyCoord(ra=7.53, dec=45.81, unit="deg")
confidence_level = 0.95
n_sim_bkg, n_sim_flux = 5000, 1000
threshold_percent_gw = np.flip(np.sort([0.95, 0.5]))  # GW probability contours
file_lvk = str(gwpaths.GW_INPUT_DIR / f"{source_name}.fits")

# --- Injected / assumed spectrum -------------------------------------------
# One definition, used everywhere: by the injector, by the estimators, and by
# every flux <-> luminosity conversion. `spectral_index` is the POSITIVE photon
# index of dN/dE = phi0 (E/E0)^-Gamma, so the "PWL index -2" spectrum is 2.0.
spectral_index = 2.0
e_ref          = 1.0 * u.TeV

# --- Monte-Carlo seeding ----------------------------------------------------
# Common random numbers: the same background realisations AND the same injected
# sky positions / distances are reused for every flux or luminosity tested
# during a bisection. Only the injected brightness changes between steps, so the
# "fraction above target" curve is smooth in the injected quantity instead of
# jittering by MC noise, which is what lets the bisection converge in ~10 steps.
seed_fake      = None    # None -> realisation i uses seed i
seed_positions = 12345   # sky-position draw (2D)
seed_sampling  = 12345   # sky x distance draw (3D)

# --- Iterative-UL speed (Part 2): parallelisation and n_sim schedule -------
# N_JOBS: worker processes for EACH bisection step's n_sim realisations.
#   1  -> sequential (default; identical Lambda sample to N_JOBS != 1, just slower)
#   -1 -> every available CPU
# A step never returns until all of its n_sim realisations have completed --
# sequential or parallel -- so the bisection never advances on a partial
# sample, whatever N_JOBS is.
#
# USE_DYNAMIC_N_SIM: False keeps n_sim fixed at n_sim_flux / n_sim_lum
# throughout (default, unchanged behaviour). True ramps it up as the bracket
# narrows (see `simulate.geometric_n_sim_schedule`) -- the early, coarse
# steps only need the sign of frac - CL right, so they can be cheap; only the
# last few steps, which set the quoted number, need the full sample size.
N_JOBS = -1
USE_DYNAMIC_N_SIM = False

### <span style="color:blue">Reading GW data</span>

In [ ]:
%%time
# --- Grid ---
ra_bins  = np.linspace(-180, 180, resolution_hp_to_grid)
dec_bins = np.linspace( -90,   90, resolution_hp_to_grid)
ra_grid, dec_grid = np.deg2rad(np.meshgrid(ra_bins, dec_bins))

# --- GW Map ---
(lvk_prob_hp, lvk_distmu_hp, lvk_distsigma_hp, lvk_distnorm_hp), meta_lvk_hp = read_sky_map(file_lvk, distances=True)
if USE_DIRAC_DELTA:
    nside        = hp.get_nside(lvk_prob_hp)
    lvk_prob_hp = utils.make_dirac_delta_hp(delta_coord, nside)
    lvk_prob_2d = utils.make_dirac_delta_2d(delta_coord, ra_bins, dec_bins)
else:
    lvk_prob_2d = utils.healpix2map(lvk_prob_hp, ra_bins, dec_bins)
lvk_prob_2d_smooth = gaussian_filter(lvk_prob_2d, sigma=2)

# --- Contours & Hotspot ---
threshold_maps = None if USE_DIRAC_DELTA else utils.get_hp_map_thresholds(lvk_prob_hp, threshold_percent_gw)
hotspot_coord  = utils.get_2d_map_hotspot(lvk_prob_2d, ra_bins, dec_bins)

### Defining the geometry (FoV, Energy axis, Spatial Binning)

In [ ]:
# --- Axes ---
axis_offset      = MapAxis.from_bounds(o_min, o_max, nbin=o_bins, name="offset")
axis_energy      = MapAxis.from_energy_bounds(e_min,   e_max,   nbin=e_bins, per_decade=True, name="energy")
axis_energy_true = MapAxis.from_energy_bounds(e_t_min, e_t_max, nbin=e_bins, per_decade=True, name="energy_true")
print("Energy axis edges:"); display(axis_energy.edges)
print("Offset axis edges:"); display(axis_offset.edges)

# --- Geometry ---
npix = (int(geom_width[0] / binsz), int(geom_width[1] / binsz))
geom = WcsGeom.create(
    skydir=(source_coord.ra.degree, source_coord.dec.degree),
    npix=npix, binsz=binsz, width=geom_width, frame="icrs", proj="AIR", axes=[axis_energy],
)
geom_image = geom.to_image()

# --- Coords ---
bin_c_ra, bin_c_dec, bin_edges_ra, bin_edges_dec, \
bin_area, coord_array, coord_center, separations_map = utils.extract_geom_coords(geom)

# --- Plot ---
plotting.summary_geometry(geom, bin_edges_ra, bin_edges_dec, size_fov, lvk_prob_2d, threshold_maps, correlation_radius)

### <span style="color:blue">Converting the GW information into the WCS geometry</span>

In [ ]:
%%time
# --- HEALPix Setup ---
pix_indices, nside = np.arange(len(lvk_prob_hp)), hp.npix2nside(len(lvk_prob_hp))
dec_hp, ra_hp      = utils.IndexToDeclRa(pix_indices, nside)
hp_area            = hp.nside2pixarea(nside)

# --- GW Probability on WCS ---
if USE_DIRAC_DELTA:
    prob_gw_integrated, num_hp_pixels = utils.integrate_dirac_delta_on_wcs(
        lvk_prob_hp, dec_hp, ra_hp, bin_edges_ra, bin_edges_dec)
else:
    prob_gw_integrated, num_hp_pixels = utils.integrate_hp_on_wcs(
        lvk_prob_hp, dec_hp, ra_hp, hp_area, bin_edges_ra, bin_edges_dec, bin_area)

norm_factor_gammapy_region = np.sum(prob_gw_integrated.ravel())
print(f"Geometry covers {norm_factor_gammapy_region * 100:.2f}% of the GW")

prob_gw = np.clip(prob_gw_integrated, a_min=1e-20, a_max=np.inf)
log_gw  = 2 * np.log(prob_gw)

# --- Threshold Masks ---
masks_thresholds = utils.compute_threshold_masks(
    threshold_maps, ra_grid, dec_grid, lvk_prob_2d, bin_c_ra, bin_c_dec, prob_gw_integrated
) if threshold_maps is not None else [prob_gw_integrated == 1.0]
mask_threshold_95 = masks_thresholds[0]

# --- Plot ---
plotting.summary_gw_map(geom, bin_edges_ra, bin_edges_dec, prob_gw)

### <span style="color:blue">3D GW: distance CDF per WCS bin</span>
For each WCS bin we collect the HEALPix pixels it contains and build the mixture
distance CDF with `ligo.skymap.distance.marginal_cdf`, weighting each pixel's
conditional PDF by its 2D probability. The result is cached as a lookup table in
`./data/gw_input/`.

In [ ]:
%%time
# --- Distance grid ---
# r_min must be STRICTLY POSITIVE. The 3D injector converts (L0, d) -> flux via
# 1 / d^2, so a grid that includes d = 0 lets the inverse-CDF sampler return
# exactly zero and inject an infinite flux, which then dominates the whole
# Lambda distribution. (`simulate.check_3d_sampling` flags this, and
# `perform_n_simulations_3d` refuses to run on it.) 1 Mpc is far below any
# plausible GW host distance, so nothing physical is lost.
r_min, r_max, n_r = 1 * u.Mpc, 10000 * u.Mpc, 1000
r_grid = np.linspace(r_min.value, r_max.value, n_r)   # Mpc
NORMALIZE_CDF = True          # True -> p(r | bin);  False -> joint with sky prob

cache_key  = utils.distance_cdf_cache_key(
    file_lvk, nside, bin_edges_ra, bin_edges_dec, r_grid,
    normalize=NORMALIZE_CDF, dirac=USE_DIRAC_DELTA)
file_cdf   = str(gwpaths.GW_INPUT_DIR / f"{source_name}_distcdf_{cache_key}.npz")

# NOTE (fix): bin_pixels is needed both to build the CDF table *and* as the
# n_pix_per_bin resolution diagnostic below, so we always (re)build it here,
# instead of only inside the `else` (cache-miss) branch. Previously,
# n_pix_per_bin was set to None whenever the CDF table itself was loaded from
# the cache file, silently breaking anything downstream that used it.
# This is a cheap indexing step relative to compute_distance_cdf_table, so
# recomputing it unconditionally is not a meaningful cost.
if USE_DIRAC_DELTA:
    # single hot pixel: its own distance params define the whole bin
    hot_ipix   = np.argmax(lvk_prob_hp)
    bin_pixels = np.empty(prob_gw_integrated.shape, dtype=object)
    for i in range(bin_pixels.shape[0]):
        for j in range(bin_pixels.shape[1]):
            bin_pixels[i, j] = (np.array([hot_ipix])
                                if prob_gw_integrated[i, j] > 0 else np.array([], dtype=int))
else:
    bin_pixels = utils.map_hp_pixels_to_wcs_bins(
        dec_hp, ra_hp, bin_edges_ra, bin_edges_dec, fill_empty=True, nside=nside)

n_pix_per_bin = np.array([[bin_pixels[i, j].size for j in range(bin_pixels.shape[1])]
                          for i in range(bin_pixels.shape[0])])

cached = None if OVERWRITE_RESULTS else utils.load_distance_cdf_table(file_cdf, cache_key)
if cached is not None:
    r_grid, cdf_table, cdf_valid, meta_cdf = cached
    print(f"Loaded distance CDF table from {file_cdf}")
else:
    cdf_table, cdf_valid = utils.compute_distance_cdf_table(
        bin_pixels, lvk_prob_hp, lvk_distmu_hp, lvk_distsigma_hp, lvk_distnorm_hp,
        r_grid, normalize=NORMALIZE_CDF)

    utils.save_distance_cdf_table(
        file_cdf, r_grid, cdf_table, cdf_valid,
        meta={"key": cache_key, "source": source_name, "file_lvk": file_lvk,
              "nside": int(nside), "n_r": int(n_r),
              "r_min": float(r_grid[0]), "r_max": float(r_grid[-1]),
              "normalize": bool(NORMALIZE_CDF), "dirac": bool(USE_DIRAC_DELTA),
              "shape": list(cdf_table.shape)})
    print(f"Saved distance CDF table -> {file_cdf}")

print(f"CDF table shape {cdf_table.shape}, valid bins: {cdf_valid.sum()}/{cdf_valid.size}")
print(f"HEALPix pixels per WCS bin: min={n_pix_per_bin.min()}, "
      f"median={np.median(n_pix_per_bin):.1f}, max={n_pix_per_bin.max()}  "
      f"({(n_pix_per_bin == 0).sum()} empty bins)")
if np.median(n_pix_per_bin[cdf_valid]) < 4:
    print("WARNING: fewer than ~4 HEALPix pixels per WCS bin (median, over valid bins) -- "
          "the per-bin distance ansatz is being built from very few samples; "
          "consider increasing `resolution_hp_to_grid` or coarsening `binsz`.")

In [ ]:
# --- Quantile maps + distance-prior summary plot ---
dist_med = utils.distance_map_from_cdf(r_grid, cdf_table, cdf_valid, q=0.5)
dist_lo  = utils.distance_map_from_cdf(r_grid, cdf_table, cdf_valid, q=0.05)
dist_hi  = utils.distance_map_from_cdf(r_grid, cdf_table, cdf_valid, q=0.95)

i_hot, j_hot = np.unravel_index(np.argmax(prob_gw_integrated), prob_gw_integrated.shape)
print(f"Hottest WCS bin ({i_hot},{j_hot}): "
      f"d = {dist_med[i_hot, j_hot]:.0f} [{dist_lo[i_hot, j_hot]:.0f}, "
      f"{dist_hi[i_hot, j_hot]:.0f}] Mpc (90% CI),  cdf_valid={cdf_valid[i_hot, j_hot]}")
print("LVK header DISTMEAN/DISTSTD:",
      meta_lvk_hp.get("distmean"), meta_lvk_hp.get("diststd"))

# pdf_gw_mask: GW-weighted mixture distance PDF restricted to the 95% mask --
# the reference curve reused below by the BKG animation, since the animated
# (F=0) injected positions are drawn from that same mask.
pdf_gw_mask = plotting.plot_distance_prior_summary(
    r_grid, cdf_table, cdf_valid, prob_gw_integrated, mask_threshold_95,
    dist_med, dist_lo, dist_hi, n_pix_per_bin,
    save_path=str(gwpaths.plot_path(source_name, "summary_distance_LVK.png")),
)

# Sky-marginalised distance posterior over the FULL GW map -- the canonical
# prior every flux<->luminosity conversion in Part 2 uses (the summary table,
# the final figure, and the 3D-sampler crosscheck), computed once here so
# they cannot drift apart into independently-derived versions of "the" prior.
cdf_marg, pdf_marg = simulate.marginal_distance_pdf(prob_gw_integrated, cdf_table, cdf_valid, r_grid)
mean_agg = np.trapezoid(r_grid * pdf_marg, r_grid)

### Reading the DL3 data and adding BKG hdu

In [ ]:
# --- Load Data Store ---
data_store_real = DataStore.from_dir(dir_dl3)
print(f"Obs IDs in directory: {data_store_real.obs_ids}\n")

# --- Add Background IRF ---
data_store_real.hdu_table.remove_rows(data_store_real.hdu_table["HDU_TYPE"] == "bkg")  # avoid duplicates on re-run
for obs_id in obs_ids:
    data_store_real = utils.add_bkg(data_store_real, obs_id, dir_dl3, dim_bkg=3, bkg_type=bkg_type)
data_store_real.hdu_table = data_store_real.hdu_table.copy()

# --- Observations ---
obs_collection_real = data_store_real.get_observations(obs_id=obs_ids, required_irf=["aeff", "edisp", "psf"])
obs_table = data_store_real.obs_table[np.isin(data_store_real.obs_table["OBS_ID"], obs_ids)]
hdu_table = data_store_real.hdu_table[np.isin(data_store_real.hdu_table["OBS_ID"], obs_ids)]

print(f"Total livetime: {obs_table['LIVETIME'].to(u.min).sum():.2f}\n")
display(obs_table); display(hdu_table)

#### Showing summaries for all the data + IRFs inside the DL3

In [ ]:
obs = obs_collection_real[0]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    print("EVENTS SUMMARY:"); obs.events.peek(); plt.show()
    print("EFFECTIVE AREA SUMMARY:"); obs.aeff.peek(figsize=(9, 2.7)); plt.show()
    print("ENERGY DISPERSION SUMMARY:"); obs.edisp.peek(figsize=(9, 2.7)); plt.show()
    print("PSF SUMMARY:"); obs.psf.peek(figsize=(9, 2.7)); plt.show()
    print("BACKGROUND SUMMARY:"); obs.bkg.peek(figsize=(7, 5)); plt.show()

#### We run the pipeline for the real data to compare

In [ ]:
# --- Makers ---
maker = MapDatasetMaker(selection=["counts", "background", "psf", "edisp", "exposure"])
maker_safe_mask = SafeMaskMaker(methods=["offset-max"], offset_max=size_fov)
maker_ring = RingBackgroundMaker(r_in=ring_r_in, width=ring_width, exclusion_mask=exclusion_mask)

# --- Empty Datasets ---
dataset_empty = MapDataset.create(geom, energy_axis_true=axis_energy_true)
dataset_stacked_real = MapDataset.create(geom, energy_axis_true=axis_energy_true)

# --- Processing Loop ---
dataset_real, bkg_models = [], []
for i, obs in enumerate(obs_collection_real):
    dataset    = maker.run(dataset_empty.copy(), obs)
    dataset    = maker_safe_mask.run(dataset, obs)
    dataset_on_off = maker_ring.run(dataset)

    bkg_models.append(dataset_on_off.background.copy())
    dataset_real.append(dataset_on_off)
    dataset_stacked_real.stack(dataset_on_off)

    plotting.summary_folded_counts(dataset_on_off, bin_c_ra, obs, i, len(obs_collection_real))

datasets_all_real = Datasets(dataset_real)

# --- Summary Plots ---
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    dataset_stacked_real.peek(figsize=(10, 6))
plotting.summary_maps(
    dataset_real[0], geom, bin_edges_ra, bin_edges_dec,
    lvk_prob_2d, threshold_maps, source_coord, source_name
)

### Defining the estimators needed
* `ExcessMapEstimator`: We use it for TS computation as is faster, and less complex. It uses a TopHat smearing for convolving the reco maps.
* `TSMapEstimator`: We use it for Flux computation. It takes more time but uses the full PSF for convolution of maps.

In [ ]:
# --- Shared Config ---
energy_edges = [axis_energy.edges[0], axis_energy.edges[-1]]
n_sigma_ul   = norm.ppf(confidence_level)

spatial_model  = PointSpatialModel(lon_0=source_coord.ra, lat_0=source_coord.dec)
spectral_model = PowerLawSpectralModel(index=2)

# --- Estimators ---
excess_estimator = ExcessMapEstimator(
    correlation_radius    = correlation_radius,
    correlate_off         = True,
    spectral_model        = spectral_model,
    energy_edges          = energy_edges,
    sum_over_energy_groups= True,
    selection_optional    = ["ul"],
)
excess_estimator.n_sigma_ul = n_sigma_ul

ts_estimator = TSMapEstimator(
    model                 = SkyModel(spectral_model=PowerLawSpectralModel(),
                                     spatial_model=spatial_model),
    energy_edges          = energy_edges,
    sum_over_energy_groups= True,
    selection_optional    = ["ul"],
)
ts_estimator.n_sigma_ul = n_sigma_ul

### <span style="color:blue">Crosscheck: the flux &harr; luminosity conversion</span>

Everything downstream &mdash; the 2D flux limit, the 3D luminosity limit and the
comparison between them &mdash; rests on integrating the assumed power law over the
analysis band. For $dN/dE = \phi_0 (E/E_0)^{-\Gamma}$:

$$F_{\rm ph}=\phi_0 E_0\!\!\int_{x_1}^{x_2}\!\! x^{-\Gamma}dx,\qquad
F_E=\phi_0 E_0^2\!\!\int_{x_1}^{x_2}\!\! x^{1-\Gamma}dx,\qquad
L = 4\pi d_L^2 F_E (1+z)^{\Gamma-2}$$

with $x=E/E_0$. Two things are easy to get wrong and are checked explicitly here:

* $\Gamma = 2$ &mdash; the index this analysis uses &mdash; is **exactly the singular
  case** of the energy-flux integral. The $1/(2-\Gamma)$ form returns `inf`/`nan`
  there; it must be evaluated as $\phi_0 E_0^2\ln(E_2/E_1)$.
* $L$ is the isotropic-equivalent luminosity **in the analysis band**, not a
  bolometric luminosity. The previous version of `amplitude_from_luminosity`
  computed the right number but documented it as bolometric, which silently
  changes what the quoted limit means.

The $K$-correction $(1+z)^{\Gamma-2}$ is identically 1 at $\Gamma=2$, so band
luminosities are redshift-independent at this index &mdash; a useful anchor, verified
below along with round trips, quadrature and gammapy's own model.

In [ ]:
if RUN_CROSSCHECKS:
    simulate.check_spectral_conversions(
        energy_edges = energy_edges,
        indices      = (1.5, 2.0, 2.5),
        amplitude    = 1e-12,
        distance     = 300 * u.Mpc,
    )

# What one amplitude actually means in this band, integrated properly:
print()
print(simulate.flux_point_from_amplitude(1e-12, energy_edges, index=spectral_index, e_ref=e_ref))

### Computing $ \Lambda$ for the real data

Instead of computing the UL (just for following same procedure in the simulations) we first compute the "stats", that is much faster. This include TS and the related quantities.

We compute our main parameter $\Lambda$, that is defined as the maximum TS' in the 95% region.

In [ ]:
stats_real = convolved_map_dataset_counts_statistics(
    convolved_maps = _get_convolved_maps(
        dataset = dataset_stacked_real,
        kernel = excess_estimator.estimate_kernel(dataset_stacked_real),
        mask = excess_estimator.estimate_mask_default(dataset_stacked_real),
        correlate_off = excess_estimator.correlate_off
    ),
    stat_type = "cash" # "wstat" or "cash", cash works with OnOffMapDataset
)

In [ ]:
# --- Cash Statistics ---
n_on   = stats_real.n_on.sum(axis=0)
mu_bkg = stats_real.mu_bkg.sum(axis=0)

lik_alt  = cash(n_on, n_on)
lik_null = cash(n_on, mu_bkg)
ts_sign  = np.where((n_on - mu_bkg) >= 0.0, +1.0, -1.0)
ts       = np.where((lik_null - lik_alt) < 0.0, 0.0, lik_null - lik_alt) * ts_sign
ts2      = ts + 2 * np.log(prob_gw)

# --- GW-Weighted Maximum (95% region) ---
ts2_masked        = np.where(mask_threshold_95, ts2, np.nan)
ts2_argmax        = np.unravel_index(np.nanargmax(ts2_masked), ts2_masked.shape)
lambda_real       = np.nanmax(ts2_masked)
lambda_coord_real = SkyCoord(ra=bin_c_ra[ts2_argmax], dec=bin_c_dec[ts2_argmax], unit=u.deg, frame="icrs")

plotting.summary_ts_maps(
    geom, bin_c_ra, bin_c_dec, ts, log_gw, ts2, lvk_prob_2d, threshold_maps, source_coord,
    lambda_real, lambda_coord_real, axis_energy
)

### Getting IRFs, pointing, and timing information from real data
Main goal is taking the `Datasets()` object of real data, and extract all the information:
* IRFs
* Livetime
* Starting time
* Pointings

Then we have this empty object that we can use to do our simulations repeatedly.

#### Filling the `Observation` object with the extracted data

### We create a empty unique dataset for the observations

Only thing left is set the `model` and then simulate the events `.fake()` later.

In [ ]:
# --- Load IRFs per Observation ---
obs_irfs, obs_time_ref, obs_pointing, obs_livetime = [], [], [], []
for obs_id in obs_ids:
    hdu_tab = data_store_real.hdu_table[data_store_real.hdu_table["OBS_ID"] == obs_id]
    obs_tab = data_store_real.obs_table[data_store_real.obs_table["OBS_ID"] == obs_id]

    path_dl3 = os.path.join(dir_dl3, hdu_tab[hdu_tab["HDU_NAME"] == "EVENTS"]["FILE_NAME"][0])
    path_bkg = os.path.join(dir_dl3, hdu_tab[hdu_tab["HDU_NAME"] == "BACKGROUND"]["FILE_NAME"][0])

    irfs = load_irf_dict_from_file(path_dl3)
    irfs.update(load_irf_dict_from_file(path_bkg))
    obs_irfs.append(irfs)

    t_ref  = Time(f"{obs_tab['DATE-OBS'][0]} {obs_tab['TIME-OBS'][0]}", scale="utc")
    pnt    = SkyCoord(ra=obs_tab["RA_PNT"][0], dec=obs_tab["DEC_PNT"][0], unit=u.deg, frame="icrs")
    t_live = np.sum(obs_tab["LIVETIME"])
    obs_time_ref.append(t_ref);  obs_pointing.append(pnt);  obs_livetime.append(t_live)

    print(f"\nRun {obs_id}"
          f"\n - IRFs:     {list(irfs.keys())}"
          f"\n - Livetime: {t_live/60:.2f} min"
          f"\n - Time Ref: {t_ref}"
          f"\n - Pointing: (RA={obs_tab['RA_PNT'][0]:.2f}, DEC={obs_tab['DEC_PNT'][0]:.2f}) deg")

# --- Build Observations ---
observations = [
    Observation.create(
        obs_id         = f"{i}",
        pointing       = FixedPointingInfo(obs_pointing[i]),
        livetime       = obs_livetime[i] * u.s,
        irfs           = obs_irfs[i],
        location       = observing_location,
        reference_time = obs_time_ref[i],
    ) for i in range(len(obs_irfs))
]

# --- Simulate Datasets ---
maker = MapDatasetMaker(selection=["background", "psf", "edisp", "exposure"])
maker_safe_mask = SafeMaskMaker(methods=["offset-max"], offset_max=size_fov)
dataset_empty = MapDataset.create(geom, energy_axis_true=axis_energy_true)
dataset_stacked_simulated = MapDataset.create(geom, energy_axis_true=axis_energy_true)

dataset_simulated = []
for i, obs in enumerate(observations):
    dataset = maker_safe_mask.run(maker.run(dataset_empty.copy(), obs), obs)
    dataset.background = bkg_models[i]
    dataset_simulated.append(dataset)
    dataset_stacked_simulated.stack(dataset)
    plotting.summary_folded_counts(dataset, bin_c_ra, obs, i, len(observations))

dataset_all_simulated = Datasets(dataset_simulated)
plotting.summary_maps(
    dataset_stacked_simulated, geom, bin_edges_ra, bin_edges_dec,
    lvk_prob_2d, threshold_maps, source_coord, source_name
)

### Computing correction factor for `ExcessMapEstimator`

In [ ]:
%%time
# --- Source Model ---
nsim_test     = 100
test_amplitud = 1e-8  # cm-2 s-1 TeV-1 @ 1 TeV
model_source  = Models([SkyModel(
    spatial_model  = PointSpatialModel.from_position(coord_center),
    spectral_model = PowerLawSpectralModel(index=2, amplitude=f"{test_amplitud} cm-2 s-1 TeV-1", reference="1 TeV"),
    temporal_model = ConstantTemporalModel(),
    name           = "model-simulated-test",
)])
true_flux_int = model_source[0].spectral_model.integral(
    excess_estimator.energy_edges[0], excess_estimator.energy_edges[-1]
)

# --- Containment Loop ---
containment_fluxes_test = []
for k in range(nsim_test):
    dataset_test = copy.deepcopy(dataset_stacked_simulated)
    dataset_test.models = model_source
    dataset_test.fake()
    dataset_test.models = None
    print(f"Test amplitude {test_amplitud:.1e} cm-2 s-1 TeV-1 @ 1 TeV — {k+1}/{nsim_test}", end="\r")

    result_test    = excess_estimator.run(dataset=dataset_test)
    central_index  = (0, len(result_test.flux.data[0][0])//2, len(result_test.flux.data[0].T[0])//2)
    containment_fluxes_test.append(result_test.flux.data[central_index] / (u.cm**2) / u.s / true_flux_int)

# --- Plot ---
containment_factor = np.median(containment_fluxes_test)
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(containment_fluxes_test, 18, color="darkblue", alpha=0.8)
ax.axvline(containment_factor, color="r", ls="--", label=f"Median\n{nsim_test} sim\n{containment_factor:.2f}")
ax.set(xlabel="Containment factor", ylabel="Counts", title=f"Correlation radius: {correlation_radius}")
ax.legend(frameon=False)
plt.show()

### Computting the Sky-Map ULs for real data
We compute the flux SkyMap using `ExcessEstimator`. The estimator that uses the top-hat kernel, so we need to correct those values for the containment of flux that we computed based on simulations.

In [ ]:
# --- Excess Map ---
print("Running Excess Map Estimator...")
maps_real = excess_estimator.run(dataset_stacked_real)

# --- Flux Arrays ---
def _masked(arr): return np.where(mask_threshold_95, arr, np.nan)
def _masked_array(*arrays):
    # compressed arrays: only pixels where mask_threshold_95 is True and not NaN
    flat_mask = mask_threshold_95.ravel()
    results = []
    for arr in arrays:
        flat = arr.ravel()
        valid = flat_mask & ~np.isnan(flat)
        results.append(flat[valid])
    return np.array(results)

flux_uls = maps_real["flux_ul"].data.ravel() / containment_factor
fluxs = maps_real["flux"].data.ravel()    / containment_factor
flux_uls_95 = _masked(maps_real["flux_ul"].data).ravel() / containment_factor
fluxs_95 = _masked(maps_real["flux"].data).ravel()    / containment_factor

sigs = maps_real["sqrt_ts"].data.ravel()
sigs_95 = _masked_array(maps_real["sqrt_ts"].data).ravel()

## Testing flux distributions

In [ ]:
plotting.plot_flux_ul_map_and_distributions(
    geom, bin_edges_ra, bin_edges_dec, maps_real["flux_ul"].smooth(0.5).data[0],
    source_coord, source_name, fluxs, fluxs_95, flux_uls, flux_uls_95,
    lvk_prob_2d, threshold_maps, energy_edges,
)

### Test of TS distribution

In [ ]:
_sqrt_ts_raw = np.array(maps_real["sqrt_ts"].data)
plotting.plot_significance_map_and_distribution(
    geom, bin_edges_ra, bin_edges_dec, maps_real["sqrt_ts"].smooth(0.5).data[0],
    vmin=-_sqrt_ts_raw.max(), vmax=_sqrt_ts_raw.max(),
    source_coord=source_coord, source_name=source_name, sigs=sigs, sigs_95=sigs_95,
    lvk_prob_2d=lvk_prob_2d, threshold_maps=threshold_maps,
)

### Defining the needed input parameters in the simulations

In order to perform the simulations we store the relevant information into a `.pkl` file:
* The ExcessMapEstimator
* The empty dataset to be simulated (all runs contained)
* The GW SkyMap converted into WCS map
* The mask of the Map for the 95% region
* The containment factor for the flux

In [ ]:
type_obs = "mono" if "mono" in dir_dl3.lower() else "stereo" if "stereo" in dir_dl3.lower() else "unknown"

_pkl_base = dict(
    excess_estimator   = excess_estimator,
    ts_estimator       = ts_estimator,
    prob_gw            = prob_gw,
    lvk_prob_hp        = lvk_prob_hp,
    mask_threshold_95  = mask_threshold_95,
    containment_factor = containment_factor,
    e_min              = e_min,
    e_max              = e_max,
    energy_edges       = energy_edges,   # analysis band; the 3D pipeline needs it
    spectral_index     = spectral_index, # assumed shape, stored so it cannot drift
    e_ref              = e_ref,
    r_grid             = r_grid,         # 3D: distance grid [Mpc]
    cdf_table          = cdf_table,      # 3D: per-WCS-bin marginal distance CDF
    cdf_valid          = cdf_valid,      # 3D: whether a bin has a usable CDF
    type_obs           = type_obs,
    source_name        = source_name,
)
_pkl_stem = os.path.join(str(gwpaths.TMP_DIR), gwpaths.run_stem(
    source_name, type_obs, e_min.value, e_max.value, bkg_type,
    correlation_radius.value, binsz, use_dirac_delta=USE_DIRAC_DELTA))

path_pkl      = f"{_pkl_stem}.pkl"       # simulated dataset (used downstream)
path_pkl_real = f"{_pkl_stem}_real.pkl"  # real dataset

for path, dataset in [(path_pkl, dataset_stacked_simulated), (path_pkl_real, dataset_stacked_real)]:
    with open(path, "wb") as f:
        pickle.dump({**_pkl_base, "dataset": dataset}, f)

# --- CROSSCHECK: fail here, not 5000 iterations later ----------------------
# Shape mismatches between the GW map and the WCS geometry, a transposed
# cdf_table, an empty 95% mask, a zero containment factor or a distance grid
# that truncates the posterior are all silent at write time and catastrophic
# (or, worse, quietly wrong) at simulation time.
print("\n--- Validating the stored input ---")
info_2d = simulate.validate_simulation_input({**_pkl_base, "dataset": dataset_stacked_simulated}, mode="2d")
info_3d = simulate.validate_simulation_input({**_pkl_base, "dataset": dataset_stacked_simulated}, mode="3d")

## Part 2 — Simulations

Background-only realisations first (for the observation's significance and the BKG
$\Lambda$ reference), then the two iterative (bisection) upper limits, 2D flux and 3D
luminosity. Both bisections call `gwuls.simulate.run_iterative_ul[_3d]`, which shares one
bisection engine (`simulate._bisect_ul`) and, per step, can run its `n_sim` realisations
sequentially or split across `N_JOBS` processes -- a step never returns until every one of
its realisations has finished, so the bisection never sees a partial Lambda sample.

### Background-only simulations

In [ ]:
OVERWRITE_RESULTS = False

In [ ]:
%%time
# --- Config ---
compute_uls   = False
amplitude_bkg = 0.0

_stem = gwpaths.run_stem(source_name, type_obs, e_min.value, e_max.value, bkg_type,
                         correlation_radius.value, binsz, use_dirac_delta=USE_DIRAC_DELTA)
_ext  = "extended" if compute_uls else ""
path_results_bkg = os.path.join(str(gwpaths.RESULTS_DIR), f"{_stem}_N{n_sim_bkg}_f{amplitude_bkg}_{_ext}.npz")

# --- Simulate or Load ---
if not os.path.exists(path_results_bkg) or OVERWRITE_RESULTS:
    simulate.perform_n_simulations(
        n_sim          = n_sim_bkg,
        amplitude      = amplitude_bkg,
        file_input     = path_pkl,
        file_output    = path_results_bkg,
        compute_uls    = compute_uls,
        spectral_index = spectral_index,
        e_ref          = e_ref,
        seed           = seed_fake,
        position_seed  = seed_positions,
        store_ts_maps  = True,   # needed by the per-pixel and TS' plots below
        store_stats    = True,
        n_jobs         = N_JOBS,
    )
else:
    print("Data already exists")

# --- Load Results ---
print(f"\nLoading for N={n_sim_bkg}...\n")
data_bkg = np.load(path_results_bkg, allow_pickle=True)

lambda_bkg, lambda_ra_bkg, lambda_dec_bkg = data_bkg["lambda_data"], data_bkg["lambda_ra"],  data_bkg["lambda_dec"]
tsmax_bkg, tsmax_ra_bkg, tsmax_dec_bkg    = data_bkg["tsmax"], data_bkg["tsmax_ra"], data_bkg["tsmax_dec"]
f_ra_bkg, f_dec_bkg = data_bkg["f_ra"], data_bkg["f_dec"]
ts_dist, ts2_dist   = data_bkg["ts_dist"], data_bkg["ts2_dist"]

if compute_uls:
    ulmax_bkg, ul_dist = data_bkg["ulmax"], data_bkg["ul_dist"]
    ulmax_ra_bkg, ulmax_dec_bkg = data_bkg["ulmax_ra"], data_bkg["ulmax_dec"]

# --- CROSSCHECK: is the background Lambda distribution self-consistent? ----
# With amplitude = 0 these realisations contain background only, so:
#   * Lambda should be a stable statistic -- split the sample in half and the
#     two medians should agree to within their MC error,
#   * the observed p-value has a binomial error that bounds how precisely the
#     significance can be quoted at all.
_h1, _h2 = np.array_split(np.asarray(lambda_bkg, dtype=float), 2)
_p       = np.mean(np.asarray(lambda_bkg) > lambda_real)
_p_err   = np.sqrt(max(_p * (1 - _p), 1e-12) / len(lambda_bkg))
print(f"BKG Lambda: median {np.median(lambda_bkg):.3f}  "
      f"(half-sample medians {np.median(_h1):.3f} / {np.median(_h2):.3f})")
print(f"p-value     {_p:.4f} +- {_p_err:.4f} (binomial, N={len(lambda_bkg)})  "
      f"-> significance {norm.ppf(1 - _p):+.2f} sigma "
      f"[{norm.ppf(1 - min(_p + _p_err, 0.999)):+.2f}, {norm.ppf(1 - max(_p - _p_err, 1e-4)):+.2f}]")
if _p in (0.0, 1.0):
    print("  NOTE: the p-value saturates the simulation sample; the significance is "
          f"only bounded, not measured (|sigma| > {abs(norm.ppf(1/len(lambda_bkg))):.2f}).")

Then we can compute some extra parameters of comprobations:
* `p-value` The fraction of simulations above the real value.
* `significance` Converting the p-value to fraction of the normal distribution.
* `lambda median` The median value of BKG distribution, that we will use as reference.

#### Crosscheck plot + BKG $\Lambda$ distribution

In [ ]:
# --- Statistics ---
p_value      = np.mean(lambda_bkg > lambda_real)
significance = norm.ppf(1 - p_value)
lambda_bkg_m = np.median(lambda_bkg)

# --- Sky Maps ---
def _fill_map(geom_image, lon, lat):
    m = Map.from_geom(geom=geom_image)
    m.fill_by_coord({"lon": lon * u.deg, "lat": lat * u.deg})
    return m

map_lambda_bkg = _fill_map(geom_image, lambda_ra_bkg,  lambda_dec_bkg)
map_ts_bkg     = _fill_map(geom_image, tsmax_ra_bkg,   tsmax_dec_bkg)
map_source_sim = _fill_map(geom_image, f_ra_bkg,        f_dec_bkg)
if compute_uls:
    map_ul_bkg = _fill_map(geom_image, ulmax_ra_bkg, ulmax_dec_bkg)

# --- Lambda Bins ---
nbins_lambda = 100
min_l_bkg, max_l_bkg = lambda_bkg.min(), lambda_bkg.max()
bins_lambda = np.linspace(min_l_bkg, max_l_bkg, nbins_lambda)

plotting.summary_bkg_simulations(
    geom, bin_edges_ra, bin_edges_dec, lvk_prob_2d, threshold_maps,
    lambda_bkg, lambda_real, lambda_bkg_m, p_value, significance,
    map_lambda_bkg, map_ts_bkg, map_source_sim, energy_edges, bins_lambda
)

In [ ]:
dist_anim = plotting.animate_bkg_simulations(
    geom, geom_image, bin_edges_ra, bin_edges_dec, bin_c_ra, bin_c_dec,
    r_grid, cdf_table, cdf_valid, f_ra_bkg, f_dec_bkg,
    lambda_bkg, lambda_ra_bkg, lambda_dec_bkg, lambda_real, bins_lambda,
    threshold_maps, lvk_prob_2d, energy_edges,
    path_gif=str(gwpaths.plot_path(source_name, "bkg_simulations_anim.gif")),
    sampling_seed=seed_sampling, pdf_gw_mask=pdf_gw_mask,
)

#### Plotting some individual pixel n-on and n-off

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

for ax in axes:
    random_pixel = (
        np.random.randint(0, bin_edges_ra.shape[0] - 1), np.random.randint(0, bin_edges_ra.shape[1] - 1),
    )
    n_on_real = stats_real.n_on.sum(axis=0)[random_pixel]
    n_off_real = stats_real.n_bkg.sum(axis=0)[random_pixel]
    n_on_bkg = [s.n_on.sum(axis=0)[random_pixel] for s in data_bkg["stats"]]
    bin_edges = np.arange(min(n_on_bkg), max(n_on_bkg) + 2, 1)

    ax.axvline(n_on_real, color="b", ls="-", label="n-ON real")
    ax.axvline(n_off_real, color="r", ls="--", label="n-OFF real")
    ax.hist(n_on_bkg, bins=bin_edges, color="b", alpha=0.5, label="BKG realisations")
    ax.set(title=f"Pixel: {random_pixel}", xlabel="counts (sum over E)")
    ax.legend(frameon=False, loc=2)

plt.tight_layout()
plt.show()

### 2D flux upper limit

In [ ]:
# --- Shared Config ---
amplitudes = np.logspace(-13, -10, 150)
compute_uls_f = False

precision = 0.02    # bracket width to stop on, in dex
frac_tol  = 0.005   # |frac - CL| tolerance; automatically widened to 2x the
                    # binomial MC error, since with N = n_sim_flux the noise on
                    # `frac` alone is sqrt(cl(1-cl)/N) and a tighter request can
                    # never be met (it just burns every iteration).

def _results_path(amplitude):
    ext = "extended" if compute_uls_f else ""
    stem = gwpaths.run_stem(source_name, type_obs, e_min.value, e_max.value, bkg_type,
                            correlation_radius.value, binsz, use_dirac_delta=USE_DIRAC_DELTA)
    return os.path.join(str(gwpaths.RESULTS_DIR), f"{stem}_N{n_sim_flux}_f{amplitude}_{ext}.npz")

# --- Non-Iterative: Submit Slurm Jobs (2D comparison baseline for this 3D notebook) ---
if not USE_ITERATIVE_ULS:
    for amplitude in amplitudes:
        path_results_f = _results_path(amplitude)
        if os.path.exists(path_results_f) and not OVERWRITE_RESULTS:
            print(f"Amplitude {amplitude:.2e} - already exists, skipping")
            continue
        if os.path.exists(path_results_f):
            print(f"Amplitude {amplitude:.2e} - overwriting")
        slurm.submit_simulation_job(
            n_sim_flux, amplitude, path_pkl, path_results_f,
            mode="2d", spectral_index=spectral_index,
            compute_uls=compute_uls_f, job_name=f"simulate_{source_name}",
        )

# --- Non-Iterative: Load Results & Compute UL ---
if not USE_ITERATIVE_ULS:
    lambda_f, amplitudes_found = [], []
    for amplitude in amplitudes:
        path_results_f = _results_path(amplitude)
        if not os.path.exists(path_results_f):
            continue
        lambda_f.append(np.load(path_results_f, allow_pickle=True)["lambda_data"])
        amplitudes_found.append(amplitude)
    amplitudes_found = np.array(amplitudes_found)

    # Grid UL: the amplitude at which the fraction above target crosses the CL,
    # interpolated (with its MC error) rather than snapped to the nearest grid
    # point -- the same estimator the bisection uses, so the two modes agree.
    _target = lambda_bkg_m if significance < 0 else lambda_real
    _frac   = np.array([np.mean(l > _target) for l in lambda_f])
    _n      = np.array([len(l) for l in lambda_f])
    amp_ul, _amp_lo1s, _amp_hi1s, _slope = simulate._interpolate_crossing(
        amplitudes_found, _frac, _n, confidence_level)
    if not np.isfinite(amp_ul):
        print("Warning: the amplitude grid never crosses the confidence level.")
        flux_ul = flux_diff_ul = None
    else:
        _fp_grid     = simulate.flux_point_from_amplitude(amp_ul, energy_edges, spectral_index, e_ref)
        flux_ul      = _fp_grid.photon_flux
        flux_diff_ul = _fp_grid.energy_flux
    cache = {"amp_ul": amp_ul, "flux_ul": flux_ul, "flux_diff_ul": flux_diff_ul,
             "hist_amp": list(amplitudes_found), "hist_frac": list(_frac),
             "hist_err": list(np.sqrt(_frac * (1 - _frac) / _n)),
             "x_ul_lo_1sigma": _amp_lo1s, "x_ul_hi_1sigma": _amp_hi1s,
             "lambda_cache": {f"{a:.6e}": l for a, l in zip(amplitudes_found, lambda_f)}}

# --- Iterative UL ---
else:
    _iter_stem = gwpaths.run_stem(source_name, type_obs, e_min.value, e_max.value, bkg_type,
                                  correlation_radius.value, binsz, use_dirac_delta=USE_DIRAC_DELTA)
    _iter_cache = os.path.join(
        str(gwpaths.TMP_DIR),
        f"{_iter_stem}_iterative_nsim{n_sim_flux}_prec{precision}_fractol{frac_tol}.pkl",
    )

    if os.path.exists(_iter_cache) and not OVERWRITE_RESULTS:
        print("Loading iterative UL cache...")
        with open(_iter_cache, "rb") as f:
            cache = pickle.load(f)
    else:
        _n_sim_2d = (simulate.geometric_n_sim_schedule(max(50, n_sim_flux // 8), n_sim_flux)
                     if USE_DYNAMIC_N_SIM else n_sim_flux)
        cache = simulate.run_iterative_ul(
            path_pkl       = path_pkl,
            lambda_real    = lambda_real,
            lambda_bkg     = lambda_bkg,
            lambda_bkg_m   = lambda_bkg_m,
            significance   = significance,
            p_value        = p_value,
            energy_edges   = energy_edges,
            cl             = confidence_level,
            n_sim          = _n_sim_2d,
            precision      = precision,
            frac_tol       = frac_tol,
            amp_lo         = 1e-13,
            amp_hi         = 1e-10,
            max_iter       = 20,
            spectral_index = spectral_index,
            e_ref          = e_ref,
            seed           = seed_fake,
            position_seed  = seed_positions,
            check_bracket  = True, # verify the crossing is inside [amp_lo, amp_hi]
            make_plots     = True,
            n_jobs         = N_JOBS,
        )
        with open(_iter_cache, "wb") as f:
            pickle.dump(cache, f)
        print(f"Saved iterative UL cache -> {_iter_cache}")

    amp_ul       = cache["amp_ul"]
    flux_ul      = cache["flux_ul"]
    flux_diff_ul = cache["flux_diff_ul"]
    amplitudes   = np.array([float(k) for k in cache["lambda_cache"].keys()])
    lambda_f     = [cache["lambda_cache"][k] for k in cache["lambda_cache"]]

result_2d = cache   # canonical name used by the summary section below

In [ ]:
!squeue -u $USER

#### Reading the result `.npz` files (grid-scan only)

In [ ]:
OVERWRITE_RESULTS = True

In [ ]:
%%time
# Only the grid scan needs this: the iterative cache above already holds every
# Lambda sample it produced (`cache["lambda_cache"]`), so re-deriving the same
# thing here would be dead work on the default (USE_ITERATIVE_ULS = True) path.
if not USE_ITERATIVE_ULS:
    _stem_grid = gwpaths.run_stem(source_name, type_obs, e_min.value, e_max.value, bkg_type,
                                  correlation_radius.value, binsz, use_dirac_delta=USE_DIRAC_DELTA)
    cache_file = os.path.join(str(gwpaths.TMP_DIR), f"{_stem_grid}_N{n_sim_flux}_cache.pkl")

    if os.path.exists(cache_file) and not OVERWRITE_RESULTS:
        print(f"Loading cached results...\n")
        with open(cache_file, "rb") as f:
            (
                ra_f, dec_f, lambda_f, lambda_ra_f, lambda_dec_f, ts_dist_f, ts2_dist_f, tsmax_f, tsmax_ra_f, tsmax_dec_f
            ) = pickle.load(f)
    else:
        print("Reading and re-agruping data...")
        ra_f, dec_f, lambda_f, lambda_ra_f, lambda_dec_f = [], [], [], [], []
        ts_dist_f, ts2_dist_f, tsmax_f, tsmax_ra_f, tsmax_dec_f = [], [], [], [], []

        for i, amplitude in enumerate(amplitudes):
            print(f"Loading data... {i/len(amplitudes)*100:.2f}%", end="\r")

            fname_results_f = f"{_stem_grid}_N{n_sim_flux}_f{amplitude}_{'extended' if bool(compute_uls_f) else ''}.npz"
            path_results_f   = os.path.join(str(gwpaths.RESULTS_DIR), fname_results_f)
            data = np.load(path_results_f)
            _lambda_ra_f_, _lambda_dec_f_ = -data["f_ra"], data["f_dec"]

            if threshold_maps is not None:
                contour_set = plt.contour(
                    np.rad2deg(ra_grid), np.rad2deg(dec_grid), np.flip(lvk_prob_2d, axis=1), levels=[threshold_maps[0]]
                )
                paths = contour_set.get_paths()
                plt.close()
                is_inside = np.array([
                    any(p.contains_point((_lambda_ra_f_[j], _lambda_dec_f_[j])) for p in paths) for j in range(len(_lambda_ra_f_))
                ])
            else:
                is_inside = np.array([
                    mask_threshold_95[
                        np.argmin(np.abs(bin_c_dec[:,0] - data["lambda_dec"][j])),
                        np.argmin(np.abs(bin_c_ra[0,:]  - data["lambda_ra"][j]))
                    ] for j in range(len(data["lambda_dec"]))
                ])

            ra_f.append(_lambda_ra_f_[is_inside]);        dec_f.append(_lambda_dec_f_[is_inside])
            lambda_f.append(data["lambda_data"][is_inside])
            lambda_ra_f.append(data["lambda_ra"][is_inside]); lambda_dec_f.append(data["lambda_dec"][is_inside])
            tsmax_f.append(data["tsmax"][is_inside])
            tsmax_ra_f.append(data["tsmax_ra"][is_inside]); tsmax_dec_f.append(data["tsmax_dec"][is_inside])

        print()
        with open(cache_file, "wb") as f:
            pickle.dump(
                (ra_f, dec_f, lambda_f, lambda_ra_f, lambda_dec_f, ts_dist_f, ts2_dist_f, tsmax_f, tsmax_ra_f, tsmax_dec_f),
                f
            )

##### Plotting all the distributions (grid-scan only)

In [ ]:
if not USE_ITERATIVE_ULS:
    log_amplitudes = np.log10(amplitudes)
    log_edges_inner = (log_amplitudes[1:] + log_amplitudes[:-1]) / 2
    amplitude_edges_inner = 10**log_edges_inner

    first_edge = 10**(2*log_amplitudes[0] - log_edges_inner[0])
    last_edge = 10**(2*log_amplitudes[-1] - log_edges_inner[-1])

    amplitude_edges = np.concatenate([[first_edge], amplitude_edges_inner, [last_edge]])

    delta_l_bkg = max_l_bkg - min_l_bkg
    lambda_bins = np.linspace(min_l_bkg - delta_l_bkg * 0.1, max_l_bkg  +  delta_l_bkg * 1.5, 200)
    lambda_bins_ext = np.linspace(min_l_bkg - delta_l_bkg * 0.1, max_l_bkg  +  delta_l_bkg * 1.5, 200)
    lambda_bins_c = (lambda_bins[1:] + lambda_bins[:-1]) / 2

    lambda_hist_bkg = np.histogram(lambda_bkg, lambda_bins, density=False)[0]
    lambda_hist_f   = np.array([np.histogram(l, lambda_bins, density=False)[0] for l in lambda_f])

    lambda_f_frac_bkg  = np.array([np.sum(l > lambda_bkg_m) / len(l) for l in lambda_f])
    lambda_f_frac_real = np.array([np.sum(l > lambda_real)  / len(l) for l in lambda_f])

    # We check manually where the threshold point is
    amplitude95_bkg  = utils.find_amplitude_at_cl(np.log10(amplitudes), lambda_f_frac_bkg, cl=confidence_level)
    amplitude95_real = utils.find_amplitude_at_cl(np.log10(amplitudes), lambda_f_frac_real, cl=confidence_level)

    plotting.plot_grid_lambda_heatmap(lambda_bins, amplitude_edges, lambda_hist_f, lambda_hist_bkg, energy_edges)

#### Computing sensitivity + ULs

In [ ]:
if not USE_ITERATIVE_ULS:
    lambda_f_m = np.array([np.percentile(l, 50) for l in lambda_f])

    # The fraction of lambdas simulated larger than the data or the BKG of lambda
    lambda_f_frac_bkg  = np.array([np.sum(l > lambda_bkg_m) / len(l) for l in lambda_f])
    lambda_f_frac_real = np.array([np.sum(l > lambda_real)  / len(l) for l in lambda_f])

    # Fitting a sigmoid to the fraction of lambdas
    x_bkg, y_bkg = np.log10(amplitudes), lambda_f_frac_bkg
    popt_bkg, pcov_bkg   = curve_fit(
        utils.sigmoid, x_bkg, y_bkg, p0=[max(y_bkg), np.median(x_bkg), 1, min(y_bkg)]
    )
    x_real, y_real = np.log10(amplitudes), lambda_f_frac_real
    popt_real, pcov_real = curve_fit(
        utils.sigmoid, x_real, y_real, p0=[max(y_real), np.median(x_real), 1, min(y_real)]
    )

    # We check manually where the threshold point is
    amplitude95_bkg  = utils.find_amplitude_at_cl(np.log10(amplitudes), lambda_f_frac_bkg, cl=confidence_level)
    amplitude95_real = utils.find_amplitude_at_cl(np.log10(amplitudes), lambda_f_frac_real, cl=confidence_level)

    idx_f_bkg  = np.abs(amplitudes - amplitude95_bkg).argmin()
    idx_f_real = np.abs(amplitudes - amplitude95_real).argmin()

    plotting.plot_grid_crossing_curve(amplitudes, lambda_f_frac_bkg, lambda_f_frac_real,
                                      amplitude95_bkg, amplitude95_real, confidence_level, energy_edges)

else:
    plotting.plot_ul_convergence(
        cache["hist_amp"], cache["hist_frac"], cache.get("hist_err"),
        x_ul=amp_ul, cl=confidence_level,
        x_ul_lo_1sigma=cache.get("x_ul_lo_1sigma"), x_ul_hi_1sigma=cache.get("x_ul_hi_1sigma"),
        label="\\phi_0", unit="cm$^{-2}$s$^{-1}$TeV$^{-1}$", method_label="2D",
        converged=cache.get("converged"), bracket_width_dex=cache.get("bracket_width_dex", float("nan")),
        n_monotonicity_violations=cache.get("n_monotonicity_violations", 0),
        at_bracket_edge=cache.get("at_bracket_edge", False), bracket_warning=cache.get("bracket_warning"),
    )

##### Summary of results

In [ ]:
print(f"Analysis of {type_obs} data in energies {energy_edges[0]:.2f}-{energy_edges[1]:.2f}")
print(f"Significance of observations: {significance:+.2f} sigma "
      f"({'under' if significance < 0 else 'over'}-fluctuation -> limit referenced to "
      f"{'the BKG median' if significance < 0 else 'the observed'} Lambda)\n")

if not USE_ITERATIVE_ULS:
    fp_bkg  = simulate.flux_point_from_amplitude(amplitude95_bkg,  energy_edges, spectral_index, e_ref)
    fp_real = simulate.flux_point_from_amplitude(amplitude95_real, energy_edges, spectral_index, e_ref)

    col_sens = "33" if significance >= 0.0 else "32"
    col_ul   = "32" if significance >= 0.0 else "31"
    print(f"\033[{col_sens}mSensitivity (BKG median crossing)\033[0m")
    print("  " + str(fp_bkg).replace("\n", "\n  "))
    print(f"\033[{col_ul}mUpper limit (real data crossing)\033[0m")
    print("  " + str(fp_real).replace("\n", "\n  "))

    flux_point_2d = fp_bkg if significance < 0 else fp_real
    flux_bkg,  flux_diff_bkg  = fp_bkg.photon_flux,  fp_bkg.energy_flux
    flux_real, flux_diff_real = fp_real.photon_flux, fp_real.energy_flux

else:
    flux_point_2d = cache.get("flux_point") or simulate.flux_point_from_amplitude(
        amp_ul, energy_edges, spectral_index, e_ref)
    print("\033[32m2D FLUX UPPER LIMIT\033[0m")
    print("  " + str(flux_point_2d).replace("\n", "\n  "))
    if np.isfinite(cache.get("x_ul_lo_1sigma", np.nan)):
        print(f"  phi0 MC 1 sigma: [{cache['x_ul_lo_1sigma']:.3e}, "
              f"{cache['x_ul_hi_1sigma']:.3e}] cm-2 s-1 TeV-1")
    flux_real, flux_diff_real = flux_point_2d.photon_flux, flux_point_2d.energy_flux

final_f_ul = flux_point_2d.photon_flux

# --- CROSSCHECK: the global (Monte-Carlo) UL against the per-pixel sky-map UL --
# These are different statistics -- the MC limit is on the *maximum* of TS' over
# the whole 95% region and therefore includes the trials factor, whereas the
# ExcessMapEstimator UL is a per-pixel limit with no trials correction. The MC
# limit should therefore land at or above the sky-map maximum; a value well
# below it would mean the trials penalty has gone missing somewhere.
flux_ul_95_max = np.nanmax(flux_uls_95)
flux_ul_max    = np.nanmax(flux_uls)
_v = final_f_ul.value
print(f"\nUL vs sky-map 95% max: {_v:.3e} vs {flux_ul_95_max:.3e}  "
      f"({(_v - flux_ul_95_max) / flux_ul_95_max * 100:+.1f}%)")
print(f"UL vs sky-map max:     {_v:.3e} vs {flux_ul_max:.3e}  "
      f"({(_v - flux_ul_max) / flux_ul_max * 100:+.1f}%)")
if _v < flux_ul_95_max:
    print("  NOTE: the global limit is BELOW the per-pixel sky-map maximum. The global "
          "limit accounts for the trials factor over the 95% region, so it is normally "
          "the looser of the two -- worth checking the containment factor and that both "
          "assume the same spectral index.")

#### Showing the threshold simulated case (grid-scan only)

In [ ]:
if not USE_ITERATIVE_ULS:
    pwl_sim = PowerLawSpectralModel(
        amplitude=amplitudes[idx_f_bkg if significance < 0 else idx_f_real] * u.Unit("TeV-1 s-1 cm-2"), index=2
    )
    flux_sim = pwl_sim.integral(*energy_edges)

    plotting.plot_grid_threshold_case(
        lambda_bkg, lambda_bins, lambda_f[idx_f_bkg if significance < 0 else idx_f_real],
        lambda_bins_ext, lambda_real, lambda_bkg_m, p_value, significance, flux_sim, energy_edges,
    )

#### Showing the ULs in front of the real distributions

In [ ]:
plotting.plot_uls_vs_real_distribution(
    flux_uls, flux_uls_95,
    flux_real=(flux_real.value if hasattr(flux_real, "value") else float(flux_real)),
    energy_edges=energy_edges, significance=significance,
    flux_bkg=(flux_bkg.value if not USE_ITERATIVE_ULS else None),
)

#### Showing UL distributions in the simulations (grid-scan only)

In [ ]:
if compute_uls and not USE_ITERATIVE_ULS:
    flux_ul = flux_bkg if significance < 0 else flux_real
    plotting.plot_grid_ul_distributions(ulmax_bkg, flux_uls_95, ul_dist, mask_threshold_95,
                                        flux_ul.value, energy_edges)

In [ ]:
if compute_uls and not USE_ITERATIVE_ULS:
    ts2_masked_flat = np.where(mask_threshold_95, ts2, np.nan).ravel()
    plotting.plot_grid_ts_distributions(lambda_bkg, ts2_masked_flat, ts2_dist, mask_threshold_95, energy_edges)

### 3D luminosity upper limit

Conceptually the same construction as the 2D flux limit, but instead of fixing a
flux and sampling only a sky position inside the 95% region, we fix an
**isotropic-equivalent band luminosity $L_0$** and, for each realisation:

1. draw a WCS bin from the **full, unmasked** GW sky probability (`prob_gw`),
2. draw a distance from that bin's marginal distance CDF (`cdf_table`, built above),
3. convert $(L_0, d)$ into an amplitude by **integrating the power law over the
   analysis band**, $\phi_0 = L_0 / [4\pi d_L^2 (1+z)^{\Gamma-2}] / (E_0^2 \int x^{1-\Gamma}dx)$,
4. inject and `fake` a source with that amplitude at the bin centre.

The result is a $\Lambda$ distribution for that $L_0$, exactly as the 2D method
gives one for a fixed flux &mdash; only marginalised over the GW sky $\times$ distance
posterior rather than conditioned on a single flux. The $L_0$ upper limit comes
from the **same bisection engine** as the 2D limit (`simulate._bisect_ul`), so the
two share convergence criteria, the crossing estimator and the MC error treatment,
and the same `N_JOBS` / `USE_DYNAMIC_N_SIM` knobs from Part 2's intro apply here too.

**What $L_0$ means.** It is the luminosity in the analysis band
$[E_{\min}, E_{\max}]$, *not* a bolometric luminosity. That distinction matters
when comparing to the literature: a bolometric number would depend on an
assumed emission range far outside what these data constrain. Set
`luminosity_band_3d` if you need the limit in a different (e.g. 0.1&ndash;100 TeV)
reference band.

In [ ]:
# --- 3D (luminosity) config ---
n_sim_lum          = n_sim_flux   # simulations per L0 bisection step
precision_3d       = 0.02         # bracket width to stop on, in dex
frac_tol_3d        = 0.005        # widened automatically to 2x the binomial MC error
lum_lo, lum_hi     = 1e42, 1e50   # erg/s bisection bracket (verified before use)
max_iter_3d        = 20

# Band in which L0 is defined. `None` -> the analysis band, which is what makes
# the 3D limit directly comparable with the 2D one. Point it at a fixed
# reference band (e.g. [0.1, 100] TeV) only if you need a literature-comparable
# number, and say so when quoting it.
luminosity_band_3d = None
apply_k_correction = False        # identically 1 at index 2; only matters otherwise

# Sky support of the 3D draw.
#   False -> sample the FULL GW map. This is the point of the 3D method: the
#            limit is marginalised over all the probability, including the part
#            outside the 95% contour where Lambda is heavily penalised. It makes
#            the 3D limit LOOSER than a 2D limit converted at the same distance,
#            by construction and not by mistake.
#   True  -> sample only inside mask_threshold_95, i.e. the same sky support the
#            2D method uses. Any remaining difference between the two limits is
#            then purely the distance marginalisation. Useful as a crosscheck;
#            rerun the bisection with it to isolate the two effects.
restrict_to_mask_3d = False

_band_3d = energy_edges if luminosity_band_3d is None else luminosity_band_3d
print(f"L0 will be defined in [{_band_3d[0]:.2f}, {_band_3d[-1]:.2f}] "
      f"with index {spectral_index}, K-correction {'on' if apply_k_correction else 'off'}")

#### Crosscheck: the 3D sampler itself

The cells further up validated the *inputs* &mdash; that `prob_gw` &times; `cdf_table`
reproduces `ligo.skymap`'s distance posterior, and that a reference
implementation of the joint draw matches it. What follows validates the
**production sampler**, i.e. the exact function
`simulate.sample_sky_and_distance` that `perform_n_simulations_3d` calls, rather
than a re-implementation of it in the notebook. This is the check that catches a
drift between "what the notebook believes the injector does" and what it does.

Three tests:

1. **Sky marginal** &mdash; a $\chi^2$ of the sampled per-bin frequencies against the
   renormalised `prob_gw`, on bins with enough expected counts for the test to be
   valid.
2. **Distance marginal** &mdash; a KS test of the sampled distances against the
   analytic mixture $\sum_{\rm bins} w_b\, p(d\,|\,b)$. Probability *atoms* at the
   edges of `r_grid` (from a grid that truncates the posterior) are reported
   separately instead of being allowed to saturate the KS statistic, which would
   otherwise flag a grid artefact as a sampler bug.
3. **Support** &mdash; no draw at $d \le 0$. This one matters physically: the
   conversion goes as $1/d^2$, so a single distance of zero would inject an
   infinite flux and silently dominate the $\Lambda$ distribution.

In [ ]:
if RUN_CROSSCHECKS:
    check_3d = simulate.check_3d_sampling(
        path_pkl, n_sim=20000, seed=7,
        restrict_to_mask=restrict_to_mask_3d, make_plot=True)

    # Tie it back to the canonical full-sky marginal built in Part 1
    # (cdf_marg/pdf_marg/mean_agg): the sampler's mean distance must agree
    # with it to within its own MC error.
    _mc_err = np.sqrt(np.trapezoid((r_grid - mean_agg) ** 2 * pdf_marg, r_grid) / check_3d["n_sim"])
    _delta  = abs(check_3d["d_mean"] - mean_agg)
    print(f"\n<d> sampler = {check_3d['d_mean']:.1f} Mpc   vs   analytic mixture "
          f"{mean_agg:.1f} Mpc   (difference {_delta:.1f}, MC error {_mc_err:.1f})")
    print("  " + ("consistent" if _delta < 4 * _mc_err else
                  "INCONSISTENT -- the injector is not drawing from the posterior "
                  "the earlier cells validated"))

    # <1/d^2> is the quantity the luminosity limit actually depends on. It is NOT
    # 1/<d>^2: for this posterior the difference is the entire reason a 3D limit
    # differs from a 2D limit evaluated at the median distance.
    _inv_d2_naive = 1.0 / check_3d["d_median"] ** 2
    print(f"\n<1/d^2>   = {check_3d['inv_d2_mean_Mpc-2']:.4e} Mpc^-2")
    print(f"1/d_med^2 = {_inv_d2_naive:.4e} Mpc^-2   "
          f"(ratio {check_3d['inv_d2_mean_Mpc-2'] / _inv_d2_naive:.3f})")
    print("  -> the nearest realisations carry disproportionate weight; a limit quoted "
          "at the median distance is not the distance-marginalised limit.")
    # How much of the GW probability the 3D draw can actually reach, and how
    # that compares with the 2D method's support. If these differ a lot, the two
    # limits are not the same statistic -- see `restrict_to_mask_3d` above.
    _p_full = prob_gw_integrated[cdf_valid].sum() / prob_gw_integrated.sum()
    _p_mask = prob_gw_integrated[cdf_valid & mask_threshold_95].sum() / prob_gw_integrated.sum()
    print(f"\nGW probability reachable by the 3D draw (full map)      : {_p_full*100:.1f}%")
    print(f"GW probability inside the 95% mask (2D method support)  : {_p_mask*100:.1f}%")
    print(f"Currently sampling: {'the 95% mask' if restrict_to_mask_3d else 'the full map'}")

In [ ]:
%%time
_stem_3d = gwpaths.run_stem(source_name, type_obs, e_min.value, e_max.value, bkg_type,
                            correlation_radius.value, binsz, use_dirac_delta=USE_DIRAC_DELTA)
_iter_cache_3d = os.path.join(str(gwpaths.TMP_DIR), (
    f"{_stem_3d}_iterative3d_nsim{n_sim_lum}_prec{precision_3d}_fractol{frac_tol_3d}"
    f"{'_masked' if restrict_to_mask_3d else ''}.pkl"
))

if os.path.exists(_iter_cache_3d) and not OVERWRITE_RESULTS:
    print("Loading 3D iterative UL cache...")
    with open(_iter_cache_3d, "rb") as f:
        cache_3d = pickle.load(f)
else:
    _n_sim_3d = (simulate.geometric_n_sim_schedule(max(50, n_sim_lum // 8), n_sim_lum)
                 if USE_DYNAMIC_N_SIM else n_sim_lum)
    cache_3d = simulate.run_iterative_ul_3d(
        path_pkl           = path_pkl,
        lambda_real        = lambda_real,
        lambda_bkg         = lambda_bkg,
        lambda_bkg_m       = lambda_bkg_m,
        significance       = significance,
        p_value            = p_value,
        cl                 = confidence_level,
        n_sim              = _n_sim_3d,
        precision          = precision_3d,
        frac_tol           = frac_tol_3d,
        lum_lo             = lum_lo,
        lum_hi             = lum_hi,
        max_iter           = max_iter_3d,
        spectral_index     = spectral_index,
        e_ref              = e_ref,
        seed               = seed_fake,
        sampling_seed      = seed_sampling,
        luminosity_band    = luminosity_band_3d,
        apply_k_correction = apply_k_correction,
        restrict_to_mask   = restrict_to_mask_3d,
        check_bracket      = True,
        n_jobs             = N_JOBS,
    )
    with open(_iter_cache_3d, "wb") as f:
        pickle.dump(cache_3d, f)
    print(f"Saved 3D iterative UL cache -> {_iter_cache_3d}")

result_3d       = cache_3d
lum_ul          = cache_3d["lum_ul"]
last_lum_tested = cache_3d["hist_lum"][-1]
lambda_3d_last  = cache_3d["lambda_cache"][f"{last_lum_tested:.6e}"]

#### Verifying the 3D bisection actually converged

The same two diagnostics used for the 2D limit: the convergence trace of the
bisection (now with the binomial error on each fraction, and the crossing curve
that actually defines the limit), and the full family of $\Lambda$ distributions
tested along the way. If the crossing curve is not monotonic in $L_0$ beyond the
error bars, the bisection's assumption is violated and the limit is not
trustworthy no matter how tight the bracket looks.

In [ ]:
plotting.plot_ul_convergence(
    cache_3d["hist_lum"], cache_3d["hist_frac"], cache_3d.get("hist_err"),
    x_ul=lum_ul, cl=confidence_level,
    x_ul_lo_1sigma=cache_3d.get("x_ul_lo_1sigma"), x_ul_hi_1sigma=cache_3d.get("x_ul_hi_1sigma"),
    label="L_0", unit="erg/s", method_label="3D",
    converged=cache_3d.get("converged"), bracket_width_dex=cache_3d.get("bracket_width_dex", float("nan")),
    n_monotonicity_violations=cache_3d.get("n_monotonicity_violations", 0),
    at_bracket_edge=cache_3d.get("at_bracket_edge", False), bracket_warning=cache_3d.get("bracket_warning"),
)

In [ ]:
# CROSSCHECK: (a) the injected-signal Lambda distribution must separate from BKG
# monotonically as L0 grows, and (b) the tested L0 values must actually straddle
# the target -- if they all sit on one side of it the bisection never bracketed
# the crossing and the returned "limit" is a bracket edge.
target_lambda = cache_3d["target"]
plotting.plot_lambda_distributions_by_x(
    lambda_bkg, lambda_real, lambda_bkg_m,
    {float(k): v for k, v in cache_3d["lambda_cache"].items()},
    target_lambda=target_lambda, cl=confidence_level,
    x_label="L_0", x_unit="erg s$^{-1}$",
)

#### Crosscheck: closure test at the quoted limit

The bisection returns an $L_0$ interpolated from the fraction curve, so it is not
a value that was itself simulated. The definitive test of the limit is therefore
to run a fresh, independent set of simulations **at exactly the quoted
$L_0^{\rm UL}$** (with a different seed, so it is not the same realisations
reshuffled) and confirm that the fraction of $\Lambda$ above the target comes out
at the confidence level, within the binomial error.

If it does not, the limit is wrong regardless of how cleanly the bisection
converged &mdash; typically because the fraction curve was interpolated across a
region where it is not locally linear in $\log L_0$, or because the crossing was
noise rather than signal.

In [ ]:
RUN_CLOSURE_TEST = True   # costs one extra block of n_sim_lum simulations

if RUN_CLOSURE_TEST:
    _closure_file = os.path.join(str(gwpaths.TMP_DIR), f"closure3d_{lum_ul:.6e}.npz")
    simulate.perform_n_simulations_3d(
        n_sim              = n_sim_lum,
        luminosity         = lum_ul,
        file_input         = path_pkl,
        file_output        = _closure_file,
        compute_uls        = 0,
        spectral_index     = spectral_index,
        e_ref              = e_ref,
        seed               = 900_000,          # independent background realisations
        sampling_seed      = 987_654,          # independent sky/distance draws
        luminosity_band    = luminosity_band_3d,
        apply_k_correction = apply_k_correction,
        restrict_to_mask   = restrict_to_mask_3d,
        store_ts_maps      = False,
        store_stats        = False,
        n_jobs             = N_JOBS,
    )
    _cl_data   = np.load(_closure_file, allow_pickle=True)
    lambda_clo = _cl_data["lambda_data"]
    d_clo, amp_clo = _cl_data["d_sim"], _cl_data["amp_sim"]

    _frac_clo = np.mean(lambda_clo > target_lambda)
    _err_clo  = np.sqrt(max(_frac_clo * (1 - _frac_clo), 1e-12) / len(lambda_clo))
    _pull     = (_frac_clo - confidence_level) / _err_clo

    print(f"\n{'='*70}")
    print(f"CLOSURE TEST at L0 = {lum_ul:.4e} erg/s  (N = {len(lambda_clo)}, independent seeds)")
    print(f"  fraction > target : {_frac_clo:.4f} +- {_err_clo:.4f}")
    print(f"  confidence level  : {confidence_level:.4f}")
    print(f"  pull              : {_pull:+.2f} sigma   "
          f"{'-> PASS' if abs(_pull) < 3 else '-> FAIL, the quoted limit is biased'}")
    print(f"{'='*70}")

    _photon_flux_clo = np.array([
        simulate.pwl_photon_flux(a, energy_edges, spectral_index, e_ref).value for a in amp_clo])
    plotting.plot_closure_test(
        lambda_bkg, lambda_clo, target_lambda, _frac_clo, _err_clo, confidence_level,
        d_clo, _photon_flux_clo, flux_point_2d.photon_flux.value, x_label="L_0",
    )

### Summary: both methods, as a flux limit and as a luminosity limit

The two methods measure different things, so neither is complete on its own:

* the **2D** method measures a **flux** and needs a distance before it can be
  quoted as a luminosity,
* the **3D** method measures a **luminosity** and needs a distance before it can
  be quoted as a flux.

The GW distance posterior spans a factor of a few, i.e. roughly an order of
magnitude in $L \propto d^2$, so neither conversion is a single number. Below,
each limit is therefore reported in **both** currencies, with the distance
dependence shown as a quantile range over the sky-marginalised distance posterior
(`cdf_marg` / `pdf_marg`, built once in Part 1) rather than collapsed onto one
hand-picked pixel's median distance, which understates the spread by a factor of
several.

Every flux quantity &mdash; photon flux, energy flux and the SED point $E^2 dN/dE$ &mdash;
comes from integrating the same power law over the analysis band; none of them is
an amplitude-at-1-TeV used as a stand-in for an integral.

In [ ]:
# --- The one table that holds both limits, both ways ------------------------
summary = simulate.summarize_upper_limits(
    result_2d          = result_2d,
    result_3d          = result_3d,
    energy_edges       = energy_edges,
    r_grid             = r_grid,
    distance_cdf       = cdf_marg,
    index              = spectral_index,
    e_ref              = e_ref,
    cl                 = confidence_level,
    significance       = significance,
    source_name        = source_name,
    percentiles        = (5, 50, 95),
    apply_k_correction = apply_k_correction,
)

# --- Reference conversions at single pixels, kept for continuity ------------
# These are the numbers the previous version of this notebook quoted. They are
# point estimates of a quantity with an order-of-magnitude prior spread, so they
# are reported alongside the marginalised range, not instead of it.
_bin_coords  = SkyCoord(ra=bin_c_ra * u.deg, dec=bin_c_dec * u.deg, frame="icrs")
i_ctr, j_ctr = np.unravel_index(np.argmin(_bin_coords.separation(coord_center).deg),
                                bin_c_ra.shape)
if not cdf_valid[i_ctr, j_ctr]:
    print(f"NOTE: bin ({i_ctr},{j_ctr}) at coord_center has no valid distance CDF; "
          f"using the GW hotspot bin instead.")
    i_ctr, j_ctr = i_hot, j_hot

for _label, (_i, _j) in [("injection pixel", (i_ctr, j_ctr)), ("GW hotspot", (i_hot, j_hot))]:
    _l = simulate.luminosity_from_amplitude(
        result_2d["amp_ul"], dist_med[_i, _j] * u.Mpc, energy_edges,
        index=spectral_index, e_ref=e_ref,
        redshift="auto" if apply_k_correction else None,
        apply_k_correction=apply_k_correction)
    print(f"[single-pixel reference] 2D UL at the {_label} ({_i},{_j}), "
          f"d_med = {dist_med[_i, _j]:.0f} Mpc  ->  L = {_l.value:.3e} erg/s")

#### Both limits on one figure

In [ ]:
_d_q   = summary["distance_quantiles_Mpc"]
_lum2  = summary["2d"]["luminosity_equivalent"]["luminosities"].value
_flx3  = summary["3d"]["flux_equivalent"]["photon_fluxes"].value
_flx2  = flux_point_2d.photon_flux.value
_lum3  = summary["3d"]["luminosity"]
_lum3_1sigma = (
    (result_3d["x_ul_lo_1sigma"], result_3d["x_ul_hi_1sigma"])
    if np.isfinite(result_3d.get("x_ul_lo_1sigma", np.nan)) else None
)

plotting.plot_uls_summary(
    r_grid, pdf_marg, _d_q, _lum2, _flx3, _flx2, _lum3, _lum3_1sigma,
    flux_uls_95, result_2d, result_3d, amp_ul, lum_ul, energy_edges,
    source_name, type_obs, confidence_level, spectral_index,
)

In [ ]:
# --- Machine-readable export of everything quoted above ---------------------
_results_out = {
    "source_name": source_name, "type_obs": type_obs,
    "e_min_TeV": float(energy_edges[0].to_value(u.TeV)),
    "e_max_TeV": float(energy_edges[-1].to_value(u.TeV)),
    "spectral_index": float(spectral_index),
    "e_ref_TeV": float(u.Quantity(e_ref).to_value(u.TeV)),
    "confidence_level": float(confidence_level),
    "lambda_real": float(lambda_real), "lambda_bkg_median": float(lambda_bkg_m),
    "p_value": float(p_value), "significance": float(significance),
    # 2D
    "amp_ul_2d": float(result_2d["amp_ul"]),
    "photon_flux_ul_2d": float(flux_point_2d.photon_flux.value),
    "energy_flux_ul_2d": float(flux_point_2d.energy_flux.value),
    "e2dnde_ul_2d": float(flux_point_2d.e2dnde.value),
    "e_dec_TeV": float(flux_point_2d.e_dec.to_value(u.TeV)),
    "lum_equiv_2d_p05_p50_p95": [float(x) for x in _lum2],
    # 3D
    "lum_ul_3d": float(lum_ul),
    "flux_equiv_3d_p05_p50_p95": [float(x) for x in _flx3],
    "luminosity_band_TeV": [float(u.Quantity(_band_3d[0]).to_value(u.TeV)),
                            float(u.Quantity(_band_3d[-1]).to_value(u.TeV))],
    "apply_k_correction": bool(apply_k_correction),
    # distances and convergence
    "distance_p05_p50_p95_Mpc": [float(x) for x in _d_q],
    "converged_2d": bool(result_2d.get("converged", False)),
    "converged_3d": bool(result_3d.get("converged", False)),
    "n_sim_bkg": int(n_sim_bkg), "n_sim_flux": int(n_sim_flux), "n_sim_lum": int(n_sim_lum),
    "n_jobs": int(N_JOBS), "dynamic_n_sim": bool(USE_DYNAMIC_N_SIM),
}

_out_json = os.path.join(str(gwpaths.RESULTS_DIR), f"{_stem}_upper_limits.json")
with open(_out_json, "w") as f:
    json.dump(_results_out, f, indent=2)
print(f"Saved -> {_out_json}\n")
for k, v in _results_out.items():
    print(f"  {k:32s} {v}")